# Week 42, Day 2: Doubly linked lists

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w042d2.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
class Node:
    def __init__(self, value):
        self.value = value
        self.prev = self.next = None

a, b, c = Node("a"), Node("b"), Node("c")
a.next, b.prev = b, a
b.next, c.prev = c, b

node = c
while node is not None:
    print(node.value, end=" ")
    node = node.prev
print()
b.prev.next, b.next.prev = b.next, b.prev        # take b out
print(a.next.value, c.prev.value)

In [ ]:
class Node:
    def __init__(self, value):
        self.value, self.prev, self.next = value, None, None

class Plain:
    def __init__(self):
        self.head = self.tail = None

    def append(self, value):
        node = Node(value)
        if self.tail is None:                  # special case: empty list
            self.head = self.tail = node
        else:
            node.prev = self.tail
            self.tail.next = node
            self.tail = node
        return node

    def remove(self, node):
        if node.prev is None:                  # special case: it's the head
            self.head = node.next
        else:
            node.prev.next = node.next
        if node.next is None:                  # special case: it's the tail
            self.tail = node.prev
        else:
            node.next.prev = node.prev

    def values(self):
        out, node = [], self.head
        while node is not None:
            out.append(node.value)
            node = node.next
        return out

xs = Plain()
nodes = [xs.append(v) for v in "abcd"]
xs.remove(nodes[0]); xs.remove(nodes[3]); xs.remove(nodes[1])
print(xs.values(), xs.head.value, xs.tail.value)

In [ ]:
class Node:
    def __init__(self, value=None):
        self.value = value
        self.prev = self.next = self           # a lone node points to itself

class DList:
    def __init__(self, values=()):
        self._sentinel = Node()
        self._size = 0
        for v in values:
            self.push_back(v)

    def _insert_after(self, place, value):
        node = Node(value)
        node.prev, node.next = place, place.next
        place.next.prev = node
        place.next = node
        self._size += 1
        return node

    def _unlink(self, node):
        node.prev.next = node.next
        node.next.prev = node.prev
        self._size -= 1
        return node.value

    def push_front(self, value):
        return self._insert_after(self._sentinel, value)

    def push_back(self, value):
        return self._insert_after(self._sentinel.prev, value)

    def pop_front(self):
        if self._size == 0:
            raise IndexError("pop from empty list")
        return self._unlink(self._sentinel.next)

    def pop_back(self):
        if self._size == 0:
            raise IndexError("pop from empty list")
        return self._unlink(self._sentinel.prev)

    def values(self):
        out, node = [], self._sentinel.next
        while node is not self._sentinel:
            out.append(node.value)
            node = node.next
        return out

d = DList("bc")
d.push_front("a")
d.push_back("d")
print(d.values(), d.pop_back(), d.pop_front(), d.values())

In [ ]:
class Node:
    def __init__(self, value=None):
        self.value = value
        self.prev = self.next = self

class Walk:
    def __init__(self, sentinel, forwards):
        self._sentinel = sentinel
        self._forwards = forwards
        self._node = sentinel.next if forwards else sentinel.prev

    def __iter__(self):
        return self

    def __next__(self):
        if self._node is self._sentinel:
            raise StopIteration
        value = self._node.value
        self._node = self._node.next if self._forwards else self._node.prev
        return value

class DList:
    def __init__(self, values=()):
        self._sentinel = Node()
        for v in values:
            node, last = Node(v), self._sentinel.prev
            node.prev, node.next = last, self._sentinel
            last.next = self._sentinel.prev = node

    def __iter__(self):
        return Walk(self._sentinel, True)

    def __reversed__(self):
        return Walk(self._sentinel, False)

d = DList([1, 2, 3, 4])
print(list(d), list(reversed(d)), list(zip(d, reversed(d))))

> **Key idea.** In a doubly linked list each node links to the next **and** the previous one, so both ends are O(1), the list can be walked backwards, and a node you hold can be removed in O(1); the price is a second link per node, and an invariant (`x.next is y` exactly when `y.prev is x`) that every method must keep. A sentinel node, linked into a circle, removes the special cases: every real node has neighbours, an empty list is a sentinel pointing to itself, and loops stop at the sentinel. A dictionary from key to node, beside the list, makes finding O(1) too: the basis of an LRU cache.

### Exercise: Linking both ways

`Node` has `value`, `prev` and `next`. Write `build(values)`, which makes a doubly linked list (no sentinel) and returns `(head, tail)` (both `None` for no values), and `backwards(tail)`, which returns the values from the tail back to the head. Then write `check(head)`: true if every link is matched (`node.next.prev is node` for every node with a next, and the head's `prev` is `None`).

In [ ]:
class Node:
    def __init__(self, value):
        self.value = value
        self.prev = self.next = None

def build(values):
    ___

def backwards(tail):
    ___

def check(head):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_build():\n    \"building\"\n    head, tail = build(\"abc\")\n    assert head.value == \"a\" and tail.value == \"c\" and head.next.next is tail and tail.prev.prev is head\n    assert backwards(tail) == [\"c\", \"b\", \"a\"] and build([]) == (None, None) and backwards(None) == []\n    assert check(head) and check(None)\n    head.next.next.prev = head\n    assert not check(head), \"a broken back link\"\n    h2, t2 = build([1])\n    h2.prev = h2\n    assert not check(h2)")

### Exercise: A deque with a sentinel

Write `Deque(values=())` using a circular doubly linked list with a sentinel node (the starter's `Node` points to itself when made). It needs `push_front`, `push_back`, `pop_front`, `pop_back` (both pops raise `IndexError("pop from empty deque")`), `peek_front`, `peek_back` (same error), `__len__`, `__bool__`, `__iter__`, `__reversed__` and `__repr__` like `Deque([1, 2])`. Write the inserting and unlinking once, in helper methods, and use no `if ... is None` tests anywhere.

In [ ]:
class Node:
    def __init__(self, value=None):
        self.value = value
        self.prev = self.next = self

class Deque:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_deque():\n    \"both ends\"\n    d = Deque([2, 3])\n    d.push_front(1)\n    d.push_back(4)\n    assert repr(d) == \"Deque([1, 2, 3, 4])\" and len(d) == 4 and d and list(reversed(d)) == [4, 3, 2, 1]\n    assert d.peek_front() == 1 and d.peek_back() == 4 and d.pop_back() == 4 and d.pop_front() == 1 and list(d) == [2, 3]\n    d.pop_front(); d.pop_back()\n    assert not d and repr(d) == \"Deque([])\" and list(d) == [] and list(reversed(d)) == []\n    for name in [\"pop_front\", \"pop_back\", \"peek_front\", \"peek_back\"]:\n        try:\n            getattr(d, name)()\n            assert False, name\n        except IndexError:\n            pass\n    d.push_back(\"x\")\n    assert list(d) == [\"x\"] and list(reversed(d)) == [\"x\"]\ndef test_style():\n    \"no None checks\"\n    assert \"is None\" not in source and \"is not None\" not in source\n    d = Deque()\n    for i in range(30_000):\n        d.push_front(i)\n        d.push_back(i)\n    for _ in range(29_999):\n        d.pop_front()\n        d.pop_back()\n    assert list(d) == [0, 0]")

### Exercise: Working with nodes you hold

Write `Playlist`, a doubly linked list with a sentinel whose `add(song)` returns the **node** it made, so the caller can keep it as a handle. Then: `insert_after(node, song)` (returns the new node), `remove(node)` (O(1); returns the song), `move_to_front(node)` (O(1)), and `songs()` (a list, in order). `len(p)` is the number of songs.

In [ ]:
class Node:
    def __init__(self, value=None):
        self.value = value
        self.prev = self.next = self

class Playlist:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_handles():\n    \"handles\"\n    p = Playlist()\n    a, b, c = p.add(\"Zombie\"), p.add(\"Lady\"), p.add(\"Water No Get Enemy\")\n    x = p.insert_after(a, \"Sorrow\")\n    assert p.songs() == [\"Zombie\", \"Sorrow\", \"Lady\", \"Water No Get Enemy\"] and len(p) == 4\n    assert p.remove(b) == \"Lady\" and p.songs() == [\"Zombie\", \"Sorrow\", \"Water No Get Enemy\"] and len(p) == 3\n    p.move_to_front(c)\n    assert p.songs() == [\"Water No Get Enemy\", \"Zombie\", \"Sorrow\"]\n    p.move_to_front(c)\n    assert p.songs() == [\"Water No Get Enemy\", \"Zombie\", \"Sorrow\"], \"moving the first to the front changes nothing\"\n    p.insert_after(x, \"End\")\n    assert p.songs()[-1] == \"End\" and len(p) == 4\ndef test_fast():\n    \"O(1) removals\"\n    p = Playlist()\n    nodes = [p.add(i) for i in range(40_000)]\n    for n in nodes[::2]:\n        p.remove(n)\n    for n in nodes[1:200:2]:\n        p.move_to_front(n)\n    assert len(p) == 20_000 and p.songs()[0] == 199")

### Exercise: Indexing from the nearer end

Add `__getitem__(i)` to the starter's list: like a Python list, negative `i` counts from the end, and out of range raises `IndexError`. Walk from the **front** if the item is in the first half, and from the **back** otherwise, so no lookup takes more than about `len / 2` steps. Count the steps in `self.steps` (one per link followed) so the checks can see it.

In [ ]:
class Node:
    def __init__(self, value=None):
        self.value = value
        self.prev = self.next = self

class DList:
    def __init__(self, values=()):
        self._sentinel = Node()
        self._size = 0
        self.steps = 0
        for v in values:
            node, last = Node(v), self._sentinel.prev
            node.prev, node.next = last, self._sentinel
            last.next = self._sentinel.prev = node
            self._size += 1

    def __len__(self):
        return self._size

    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_index():\n    \"indexing\"\n    d = DList(range(10))\n    assert [d[i] for i in range(10)] == list(range(10)) and d[-1] == 9 and d[-10] == 0\n    for bad in [10, -11]:\n        try:\n            d[bad]\n            assert False\n        except IndexError:\n            pass\ndef test_steps():\n    \"the nearer end\"\n    d = DList(range(1000))\n    d.steps = 0\n    d[999]\n    assert d.steps <= 1, d.steps\n    d.steps = 0\n    d[500]\n    assert d.steps <= 501\n    worst = 0\n    for i in range(1000):\n        d.steps = 0\n        d[i]\n        worst = max(worst, d.steps)\n    assert worst <= 501, worst")

### Bug hunt: Only one way round

`unlink` fixes the forward link but forgets the backward one, so walking forwards looks right but walking backwards still finds the removed node: this prints `['a', 'c']` and then `['c', 'b', 'a']`. Fix it so the second line is `['c', 'a']`.

In [ ]:
class Node:
    def __init__(self, value=None):
        self.value = value
        self.prev = self.next = self

sentinel = Node()
def push_back(value):
    node, last = Node(value), sentinel.prev
    node.prev, node.next = last, sentinel
    last.next = sentinel.prev = node
    return node

def unlink(node):
    node.prev.next = node.next

def walk(forwards):
    out, node = [], sentinel.next if forwards else sentinel.prev
    while node is not sentinel:
        out.append(node.value)
        node = node.next if forwards else node.prev
    return out

a, b, c = push_back("a"), push_back("b"), push_back("c")
unlink(b)
print(walk(True))
print(walk(False))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"both directions\"\n    expect_output(\"['a', 'c']\\n['c', 'a']\")")

### Bug hunt: Links set in the wrong order

`insert_after` changes `place.next` **before** it uses it to find the node that should come after the new one, so the new node points at itself and the rest of the list is lost: this prints `['a', 'X']`. Reorder the four assignments so it prints `['a', 'X', 'b', 'c']`.

In [ ]:
class Node:
    def __init__(self, value=None):
        self.value = value
        self.prev = self.next = self

sentinel = Node()

def insert_after(place, value):
    node = Node(value)
    place.next = node
    node.prev = place
    node.next = place.next
    node.next.prev = node
    return node

def values():
    out, node = [], sentinel.next
    count = 0
    while node is not sentinel and count < 10:
        out.append(node.value)
        node = node.next
        count += 1
    return list(dict.fromkeys(out))

c = insert_after(sentinel, "c")
b = insert_after(sentinel, "b")
a = insert_after(sentinel, "a")
insert_after(a, "X")
print(values())

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"use place.next before changing it\"\n    expect_output(\"['a', 'X', 'b', 'c']\")")

### Bug hunt: The last node leaves a ghost

Without a sentinel, `pop_back` has to handle the list becoming empty. This one forgets to clear the head when the only node goes, so the "empty" list still shows its old item: this prints `x` and then `['x']`. Fix it so the second line is `[]`.

In [ ]:
class Node:
    def __init__(self, value):
        self.value, self.prev, self.next = value, None, None

class DList:
    def __init__(self):
        self.head = self.tail = None

    def push_back(self, value):
        node = Node(value)
        if self.tail is None:
            self.head = self.tail = node
        else:
            node.prev, self.tail.next = self.tail, node
            self.tail = node

    def pop_back(self):
        node = self.tail
        self.tail = node.prev
        if self.tail is not None:
            self.tail.next = None
        return node.value

    def values(self):
        out, node = [], self.head
        while node is not None:
            out.append(node.value)
            node = node.next
        return out

d = DList()
d.push_back("x")
print(d.pop_back())
print(d.values())

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"an empty list after the last pop\"\n    expect_output(\"x\\n[]\")\n    d2 = DList()\n    for v in \"ab\":\n        d2.push_back(v)\n    d2.pop_back(); d2.pop_back(); d2.push_back(\"c\")\n    assert d2.values() == [\"c\"]")

### Puzzle: Unlinking a node

Arrange the lines into a method that removes a node from a circular list with a sentinel and returns its value. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
class Node:
    def __init__(self, value=None):
        self.value = value
        self.prev = self.next = self

s = Node()
x = Node("x")
x.prev, x.next = s, s
s.next = s.prev = x

p = ___   # s.next is s.prev
q = ___   # x.next is s
r = ___   # how many links change to remove a node from a doubly linked list with a sentinel
s_ = ___  # the cost of removing the last node of a doubly linked list: "O(1)" or "O(n)"
t = ___   # what an empty circular list's sentinel.next points to: "None" or "the sentinel"

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p is True and q is True and r == 2, \"p, q and r\"\n    assert s_ == \"O(1)\" and t == \"the sentinel\", \"s_ and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: an LRU cache

Write `LRUCache(capacity)` (a capacity below 1 raises `ValueError`) with:

- `get(key, default=None)`: the value, marking the key as **most recently used**; `default` if it's missing;
- `put(key, value)`: add or update the key, making it most recently used; if that takes the cache over capacity, remove the **least recently used** key and return it (otherwise return `None`);
- `__len__`, `__contains__` (which does **not** count as a use), and `keys()`: the keys from most to least recently used.

Every operation except `keys()` must be O(1): use a dictionary from key to node, and a doubly linked list with a sentinel, most recent at the front. Don't use `OrderedDict` or `lru_cache`.

In [ ]:
class Node:
    def __init__(self, key=None, value=None):
        self.key, self.value = key, value
        self.prev = self.next = self

class LRUCache:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_cache():\n    \"the cache\"\n    c = LRUCache(2)\n    assert c.put(\"a\", 1) is None and c.put(\"b\", 2) is None and c.get(\"a\") == 1\n    assert c.put(\"c\", 3) == \"b\" and \"b\" not in c and c.keys() == [\"c\", \"a\"]\n    assert c.get(\"b\", \"gone\") == \"gone\" and c.put(\"a\", 10) is None and c.keys() == [\"a\", \"c\"] and c.get(\"a\") == 10\n    assert \"c\" in c and c.keys() == [\"a\", \"c\"], \"in doesn't count as a use\"\n    assert c.put(\"d\", 4) == \"c\" and len(c) == 2\n    try:\n        LRUCache(0)\n        assert False\n    except ValueError:\n        pass\ndef test_one():\n    \"capacity 1\"\n    c = LRUCache(1)\n    c.put(1, \"x\")\n    assert c.put(2, \"y\") == 1 and c.keys() == [2] and c.get(1) is None\ndef test_fast():\n    \"O(1) operations\"\n    c = LRUCache(1000)\n    for i in range(100_000):\n        c.put(i % 1500, i)\n        c.get((i * 7) % 1500)\n    assert len(c) == 1000\n    assert \"OrderedDict\" not in source and \"lru_cache\" not in source")

## Warm-up

With a link each way you can walk backwards from `c`, and the last line removes `b` using only `b` itself: its neighbours are linked to each other, and nothing had to be searched.

## Two links per node

```text
           +------+------+------+      +------+------+------+      +------+------+------+
None <-----+ prev |  a   | next +----->| prev |  b   | next +----->| prev |  c   | next +-----> None
           +------+------+------+<-----+------+------+------+<-----+------+------+------+
             head                                                    tail
```

Every link is now stored twice: `x.next is y` exactly when `y.prev is x`. That's an **invariant** (Week 28) that every method must keep, and most doubly-linked-list bugs are a method that updates one direction and forgets the other.

What the second link buys:

| Operation | Singly linked (head, tail) | Doubly linked (head, tail) |
|---|---|---|
| add or remove at the front | O(1) | O(1) |
| add at the back | O(1) | O(1) |
| **remove at the back** | O(n) | **O(1)**: `tail.prev` is the new tail |
| **remove a node you hold** | O(n): find the one before it | **O(1)**: it knows both neighbours |
| walk backwards | not possible | O(n) |
| memory per node | value + 1 link | value + 2 links |

## The special cases problem

Here's `remove(node)` for a doubly linked list with a plain head and tail. Each line handles a different situation:

It works, but four branches for one operation, and the same branches again in insert, push and pop, is a lot of places to get wrong.

## A sentinel node

A **sentinel** (sometimes called a dummy or header node) is one extra node that holds no value and is never removed. Make the list **circular** through it: the sentinel's `next` is the first real node and its `prev` is the last; an empty list is a sentinel pointing to itself.

```text
             +------------------------------------------------+
             v                                                |
        [sentinel] <--> [ a ] <--> [ b ] <--> [ c ] <---------+
             ^                                   |
             +-----------------------------------+
   empty:  [sentinel] whose prev and next are both itself
```

Now **every** real node has a real node or the sentinel on both sides, so there are no `None`s to check. Inserting and removing each become two or four plain assignments:

`push_front` and `push_back` are the same operation in different places, and so are the two pops. Notice that the loop now stops at the **sentinel**, not at `None`: there are no `None`s in a circular list. (Sentinels are an old idea; Donald Knuth's *The Art of Computer Programming* uses them throughout its treatment of lists.)

## Both directions

With links both ways, a list can offer `reversed()`. Python calls `__reversed__` if a class has one, and expects an iterator back. One iterator class can walk either way, given which link to follow:

## A dictionary and a list together

The one thing a linked list can't do quickly is **find** a node by its value: that's a search, O(n). The fix is to keep a dictionary from each key to its node beside the list. The dictionary finds a node in O(1); the doubly linked list removes or moves it in O(1). Together, they keep things **in an order you can change**, with every operation O(1).

That's how a **least recently used (LRU) cache** works: a fixed number of entries, and when it's full, the one used longest ago is thrown out. Each use moves an entry to the front of the list; the least recently used is always at the back. You'll build one in today's challenge. (CPython's `collections.OrderedDict` is built the same way, a dictionary plus a doubly linked list, in its original pure-Python version; and `functools.lru_cache` is a cache of exactly this kind.)

## Practice

## Quiz

## Daily challenge

## Before you go

Draw a sentinel list with two nodes, then draw what each of `push_front`, `pop_back` and removing the first node does, arrow by arrow. Tomorrow: insertion and deletion in depth, with dummy heads for singly linked lists, removing by value, and splicing whole runs of nodes.